# Data Quality Evaluation: are the six datasets ready to combine?

Notebooks 01-06 each clean one dataset on its own. This notebook checks whether those cleaned tables **fit
together** into one player-season table for machine learning, with one row per player per season, our
features, and the label "traded in the offseason after this season."

It reads only from `data-files/02-processed/` and writes nothing. Each check prints the numbers behind it.
The verdict and the list of fixes are at the end.

**Short answer: not yet.** Each table is clean on its own, and the label join already works for about 96% of
trades. But a few cross-dataset problems would quietly corrupt a merged table: mismatched season
conventions, non-regular-season games mixed into the box scores, a mislabeled 2020 bubble playoff run,
features that leak the future, and plus-minus that was dropped. They're all fixable. The list is at the end.

In [1]:
import sys
import numpy as np
import pandas as pd

from nba_cleaning_utils import PROCESSED, normalize_name

bbref   = pd.read_csv(PROCESSED / 'bbref_team_salaries_clean.csv')            # 1
games   = pd.read_csv(PROCESSED / 'player_game_stats_cleaned.csv',           # 2
                      low_memory=False, parse_dates=['gameDate'])
kaggle  = pd.read_csv(PROCESSED / 'nba_salaries_2014_2022_clean.csv')        # 3
leaders = pd.read_csv(PROCESSED / 'nba_player_stats_cleaned.csv')            # 4
ps      = pd.read_csv(PROCESSED / 'player_season_salaries.csv')              # 5
teams   = pd.read_csv(PROCESSED / 'team_season_payrolls.csv')                # 5
trades  = pd.read_csv(PROCESSED / 'offseason_traded_players.csv')            # 6

# The modeling window from the research question: offseasons after 2015-16 ... 2024-25
WINDOW = range(2015, 2025)       # season START years

## 1. Inventory: what each table is, and how its rows are keyed

To join tables you need the same *grain* (what one row means) and the same *keys*. The table below shows that
the six sources disagree on all three keys: season, player, and team.

In [2]:
def describe(name, df, grain, season_col, convention, player_key, team_col):
    s = df[season_col]
    return {'dataset': name, 'rows': len(df), 'one row =': grain, 'season column': season_col,
            'season means': convention, 'range in file': f'{s.min()}-{s.max()}',
            'player key': player_key, 'team label': team_col,
            'team example': df[team_col].iloc[0] if team_col in df else '-'}

inventory = pd.DataFrame([
    describe('#1 BBRef salaries', bbref, 'player-team-season', 'season', 'END year', 'player_id (BBRef)', 'team'),
    describe('#2 box scores', games, 'player-game', 'season', 'END year', 'personId (NBA)', 'playerteamName'),
    describe('#3 Kaggle salaries', kaggle, 'player-season', 'seasonStartYear', 'START year', 'playerName only', '-'),
    describe('#4 league leaders', leaders, 'player-season', 'season', 'START year', 'player_id (NBA)', 'team'),
    describe('#5 player_season', ps, 'player-season', 'season_year', 'START year', 'name_key (+HoopsHype id)', 'primary_team'),
    describe('#6 offseason trades', trades, 'traded player-offseason', 'offseason_after_season', 'START year', 'player_id (NBA) + name_key', 'from_team'),
]).set_index('dataset')
inventory

,rows,one row =,season column,season means,range in file,player key,team label,team example
dataset,,,,,,,,
#1 BBRef salaries,7615,player-team-season,season,END year,2015-2026,player_id (BBRef),team,ATL
#2 box scores,355430,player-game,season,END year,2015-2027,personId (NBA),playerteamName,Trail Blazers
#3 Kaggle salaries,4466,player-season,seasonStartYear,START year,2014-2021,playerName only,-,-
#4 league leaders,2996,player-season,season,START year,2014-2025,player_id (NBA),team,OKC
#5 player_season,6317,player-season,season_year,START year,2015-2025,name_key (+HoopsHype id),primary_team,Chicago Bulls
#6 offseason trades,555,traded player-offseason,offseason_after_season,START year,2014-2025,player_id (NBA) + name_key,from_team,Milwaukee Bucks


**Season.** Two tables (#1 and #2) label a season by the year it **ends**, and the other four by the year it
**starts**. All six call the column `season`/`season_year`, so a naive `merge(on='season')` runs without an
error and silently pairs every player with the **wrong year**. The check below uses one known fact,
Stephen Curry's 2021-22 salary of $45,780,966, to show the offset.

In [3]:
curry = pd.DataFrame({
    '#1 BBRef (season = end year)':  bbref.loc[bbref['player'].eq('Stephen Curry') & bbref['salary'].eq(45780966), 'season'].tolist(),
    '#3 Kaggle (start year)':        kaggle.loc[kaggle['playerName'].eq('Stephen Curry') & kaggle['salary'].eq(45780966), 'seasonStartYear'].tolist(),
    '#5 HF (season_year = start)':   ps.loc[ps['player'].eq('Stephen Curry') & ps['total_salary'].eq(45780966), 'season_year'].tolist(),
}, index=['2021-22 season is stored as'])
curry

,#1 BBRef (season = end year),#3 Kaggle (start year),#5 HF (season_year = start)
2021-22 season is stored as,2022,2021,2021


**Fix:** before merging, add one `season_start` column to every table (`season - 1` for #1 and #2) and join
only on that.

**Team.** #1 uses Basketball-Reference codes (`BRK`, `CHO`, `PHO`), #4 uses NBA codes (`BKN`, `CHA`, `PHX`),
#2 uses only the nickname (`Lakers`), and #5 and #6 use full names. The model needs team only to attach team
payroll and cap tier (from #5) to each player, so one 30-row lookup table (code ↔ full name) is enough.

In [4]:
print('#1 BBRef team codes:', sorted(bbref['team'].unique()))
print('#4 NBA team codes:  ', sorted(leaders['team'].unique()))
print('#2 distinct playerteamName values:', games['playerteamName'].nunique(), '(30 NBA teams + All-Star / exhibition teams, see section 5)')

#1 BBRef team codes: ['ATL', 'BOS', 'BRK', 'CHI', 'CHO', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']
#4 NBA team codes:   ['ATL', 'BKN', 'BOS', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']
#2 distinct playerteamName values: 52 (30 NBA teams + All-Star / exhibition teams, see section 5)


## 2. Does the label attach to the feature rows?

The label comes from #6, and each traded player has to be found in the feature tables. #6 carries **both** the
NBA player ID (shared with #2 and #4) and a `name_key` (shared with #5), so we check both joins.

In [5]:
lab = trades[trades['offseason_after_season'].isin(WINDOW)].copy()
print('offseason trades in the window:', len(lab))

# (a) label -> #5 salary/contract features, by season + normalized name
to_ps = lab.merge(ps[['season_year', 'name_key', 'primary_team']], how='left', indicator=True,
                  left_on=['offseason_after_season', 'name_key'], right_on=['season_year', 'name_key'])
ok = to_ps['_merge'].eq('both')
print(f'(a) matched to #5 by name_key: {ok.sum()} / {len(lab)} = {ok.mean():.1%}')
print('    unmatched:', sorted(to_ps.loc[~ok, 'player'].unique()))

# (b) label -> #2 regular-season box scores, by NBA id (#2 uses end-year seasons, so shift by 1)
reg = games[games['gameType'].eq('Regular Season')]
played = reg.groupby(['season', 'personId']).size().rename('games').reset_index()
played['season_start'] = played['season'] - 1
to_box = lab.merge(played, how='left', left_on=['offseason_after_season', 'player_id'],
                   right_on=['season_start', 'personId'])
ok_b = to_box['games'].notna()
print(f'(b) matched to #2 by NBA id:    {ok_b.sum()} / {len(lab)} = {ok_b.mean():.1%}')
print('    unmatched:', sorted(to_box.loc[~ok_b, 'player'].unique()))

# Does the trade's from_team agree with #5's primary team?
both = to_ps[ok]
print(f'from_team (#6) == primary_team (#5): {(both["from_team"] == both["primary_team"]).mean():.1%}')

offseason trades in the window: 468
(a) matched to #5 by name_key: 451 / 468 = 96.4%
    unmatched: ['Ante Zizic', 'Cam Whitmore', 'Dennis Schroder', 'Enes Freedom', 'Ish Smith', 'Juancho Hernangomez', 'Lou Williams', 'Maurice Harkless', 'Ochai Agbaji', 'Patty Mills', 'Timothe Luwawu-Cabarrot', 'Toumani Camara', 'Wes Iwundu']
(b) matched to #2 by NBA id:    457 / 468 = 97.6%
    unmatched: ['Alexis Ajinca', 'Ante Zizic', 'Brandon Knight', 'Danilo Gallinari', 'Darius Miller', 'Duane Washington Jr.', 'John Wall', 'Ochai Agbaji', 'Quincy Pondexter', 'Saddiq Bey', 'Toumani Camara']
from_team (#6) == primary_team (#5): 98.9%


**What we found:**

- **The name join (a) is good but loses about 17 trades (3.6%), and they aren't random.** Most are name
  variants that `normalize_name` can't fix: HoopsHype uses legal names (*Louis* Williams, *Ishmael* Smith,
  *Enes Kanter* for Enes Freedom, *Moe* Harkless, Dennis *Schroeder*), and these are veterans who were
  traded. A small alias table (about 15 rows) mapping these `name_key`s would recover nearly all of them.
- **The NBA-ID join (b) works without any name matching.** Its misses are mostly players who didn't play that
  season (injured: John Wall, Brandon Knight) or who were drafted and traded right away, so they're expected.
  When we combine, **use the NBA ID wherever both sides have it** (#2, #4, #6), and use names only for #5.
- `from_team` and #5's primary team agree about 99% of the time. The few mismatches are players who changed
  teams during the season.

In [6]:
# The same name-variant problem, seen from the salary side: #5 players that never match #1 by name
bb_keys = set(bbref['player'].map(normalize_name))
no_match = ps.loc[~ps['name_key'].isin(bb_keys), 'player'].drop_duplicates()
print(len(no_match), '#5 players never match a #1 name, e.g.:', no_match.head(15).tolist())

60 #5 players never match a #1 name, e.g.: ['Christapher Johnson', 'Dennis Schroeder', 'Enes Kanter', 'Ishmael Smith', 'Jose Juan Barea', 'Joseph Young', 'Louis Amundson', 'Louis Williams', 'Maurice Williams', 'Moe Harkless', 'Nenê', 'Omer Asik', 'Patrick Mills', 'Scottie Wilbekin', 'Walter Tavares']


## 3. The label and the population

How rare is the outcome, and who is in the population? The research question says *players under contract for
the following season*.

In [7]:
base = ps[ps['season_year'].isin(WINDOW)].copy()
base = base.merge(lab[['offseason_after_season', 'name_key']].assign(traded=1), how='left',
                  left_on=['season_year', 'name_key'], right_on=['offseason_after_season', 'name_key'])
base['traded'] = base['traded'].fillna(0).astype(int)

summary = base.groupby('season_year').agg(player_seasons=('traded', 'size'), traded=('traded', 'sum'))
summary['trade_rate'] = (summary['traded'] / summary['player_seasons']).round(3)
display(summary.T)
print(f"overall: {base['traded'].sum()} traded of {len(base)} player-seasons = {base['traded'].mean():.1%}")

season_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
player_seasons,501.000,545.000,587.000,577.000,516.000,579.000,652.000,574.000,614.000,579.000
traded,34.000,47.000,36.000,55.000,53.000,53.000,45.000,45.000,42.000,41.000
trade_rate,0.068,0.086,0.061,0.095,0.103,0.092,0.069,0.078,0.068,0.071


overall: 451 traded of 5724 player-seasons = 7.9%


**What we found:**

- **The outcome is rare, as expected: about 7.9% of player-seasons**, roughly 45 per offseason. The
  hypothesis assumes "about 0.06" (so a PR-AUC target of 0.12). With all of #5 as the population the rate is
  higher, so **update the baseline once the final population is fixed.**
- **The population can't be defined yet without leaking the future.** "Under contract for the following
  season" has to be known at the end of season *S*. #5 has no contract length (notebook 05, concern 2). The
  only proxy is "has a salary row in *S*+1," and that row also exists for players who **re-signed or were
  signed during the same offseason** we are predicting, which uses information from after the prediction.
  The same problem affects the planned *years left on contract* feature ("count future salary rows in #5"):
  future salary rows are the realized future, including contracts signed after the trade. This is the
  biggest gap, and it needs a contract-length source (for example, Spotrac or Basketball-Reference contract
  pages captured as of each season) or an explicit, documented approximation.
- **Two columns in `player_season_salaries.csv` must never be features:** `next_season_primary_team` and
  `changed_team_next_season` describe the outcome period. A trade in the offseason directly changes them,
  so a model would score high just by reading them.

In [8]:
leaky = ['next_season_primary_team', 'changed_team_next_season']
print(pd.crosstab(base['traded'], base['changed_team_next_season'], dropna=False,
                  rownames=['traded'], colnames=['changed_team_next_season']))

changed_team_next_season   0.0   1.0   NaN
traded                                    
0                         2720  1387  1166
1                            1   433    17


## 4. Salary sources: do they agree, and can we trust 2019-20?

We have three salary sources. #5 is the main one, because it has cap lines and cap tiers. #1 and #3 were meant
as cross-checks.

In [9]:
bb = bbref.assign(season_year=bbref['season'] - 1, name_key=bbref['player'].map(normalize_name))
bb_ps = bb.groupby(['season_year', 'name_key'], as_index=False)['salary'].sum(min_count=1).dropna()

j1 = ps.merge(bb_ps, on=['season_year', 'name_key'])
rel1 = (j1['total_salary'] - j1['salary']).abs() / j1['salary']

kg = kaggle.assign(name_key=kaggle['playerName'].map(normalize_name))
j3 = ps.merge(kg, left_on=['season_year', 'name_key'], right_on=['seasonStartYear', 'name_key'],
              suffixes=('', '_kg'))
rel3 = (j3['total_salary'] - j3['salary']).abs() / j3['salary']

print(f'#5 vs #1 (BBRef):  {len(j1)} player-seasons matched, {(rel1 < .01).mean():.1%} within 1%, '
      f'{(rel1 > .10).mean():.1%} differ by >10%')
print(f'#5 vs #3 (Kaggle): {len(j3)} player-seasons matched, {(rel3 < .01).mean():.1%} within 1%')
display(j1.assign(diff=j1['total_salary'] - j1['salary'])
          .loc[rel1 > .10].nlargest(6, 'total_salary')[['season', 'player', 'total_salary', 'salary', 'diff']]
          .rename(columns={'total_salary': 'salary_#5', 'salary': 'salary_#1'}))

#5 vs #1 (BBRef):  5526 player-seasons matched, 81.3% within 1%, 11.2% differ by >10%
#5 vs #3 (Kaggle): 3950 player-seasons matched, 99.9% within 1%


,season,player,salary_#5,salary_#1,diff
3800,2022-23,John Wall,47345760,40866760.0,6479000.0
4813,2024-25,Jonathan Isaac,27600000,25000000.0,2600000.0
220,2015-16,Joe Johnson,22309344,25309344.0,-3000000.0
3694,2022-23,Domantas Sabonis,21100000,18500000.0,2600000.0
2433,2019-20,Tim Hardaway Jr,20025127,18150000.0,1875127.0
3486,2021-22,Spencer Dinwiddie,19314284,17142857.0,2171427.0


**What we found:**

- **#3 isn't an independent cross-check.** It agrees with #5 to the dollar in 99.9% of matched rows, so both
  most likely come from HoopsHype. It also stops at 2021-22. #3 adds little that #5 doesn't already have
  (only inflation-adjusted salary, and we plan to scale by the cap anyway), so we recommend keeping it as a
  documented check and **leaving it out of the merged table**.
- **#5 and #1 differ for about 19% of player-seasons.** Many large gaps are a steady $1-3M on stars on a single
  team (John Wall, Clint Capela, Domantas Sabonis). That pattern looks like *cap hit including likely
  bonuses* (HoopsHype) versus *base salary* (Basketball-Reference), not errors. Pick one definition (cap hit
  from #5, which is what cap tiers are computed from) and say so.

In [10]:
bb_payroll = bb.groupby(['season_year', 'team'])['salary'].sum().groupby('season_year').median() / 1e6
hf_payroll = teams.groupby('season_year')['payroll'].median() / 1e6
rows = pd.DataFrame({'median payroll #5 ($M)': hf_payroll.round(1), 'median payroll #1 ($M)': bb_payroll.round(1),
                     'players #5': ps.groupby('season_year').size(),
                     'players #1': bb.groupby('season_year')['player_id'].nunique()}).loc[2015:2025]
rows['teams over tax (#5)'] = teams.assign(t=teams['cap_tier'].ne('under_cap') & teams['cap_tier'].ne('over_cap')) \
                                   .groupby('season_year')['t'].sum()
rows.T

season_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
median payroll #5 ($M),76.1,98.5,115.0,121.5,120.0,130.3,135.9,150.2,165.1,170.2,188.9
median payroll #1 ($M),76.0,98.9,112.9,122.1,127.9,128.6,134.5,148.9,163.9,169.9,186.9
players #5,501.0,545.0,587.0,577.0,516.0,579.0,652.0,574.0,614.0,579.0,593.0
players #1,525.0,559.0,600.0,581.0,578.0,574.0,657.0,579.0,620.0,609.0,616.0
teams over tax (#5),9.0,3.0,9.0,11.0,0.0,10.0,11.0,15.0,14.0,13.0,18.0


**2019-20 is under-covered in #5.** It has about 60 fewer players than #1 (516 vs 578), a median payroll
about $8M below Basketball-Reference's, and **zero** teams over the tax line (every other season has
3-18). Because cap tier is one of our three main hypotheses, the 2019-20 offseason (the 2020 bubble
offseason) would get wrong tier labels. Options: rebuild 2019-20 payrolls from #1, or flag the season and
run the analysis with and without it.

## 5. Box scores (#2): what has to happen before aggregating to player-seasons

The research question uses regular-season **and** playoff minutes, points, and plus-minus per game. #2 is
the only source for playoffs and for bench players, so it has to be aggregated correctly.

In [11]:
display(games['gameType'].value_counts().rename('rows').to_frame().T)
print('non-NBA team names, e.g.:',
      sorted(set(games['playerteamName']) - set(games.loc[games['gameType'].eq('Regular Season'), 'playerteamName']))[:15])

gameType,Regular Season,Preseason,Playoffs,NBA Emirates Cup,Play-in Tournament,All-Star Game,NBA Cup,Emirates NBA Cup
rows,306314,25603,20940,1380,764,387,24,18


non-NBA team names, e.g.: ['Durant', 'East', 'Giannis', 'Global Stars', 'Jerusalem B.C.', 'LeBron', 'Lions', 'Loong-Lions', 'OGs', 'Phoenix', 'Rising Stars', 'Stars', 'Stephen', 'Stripes', 'Team Austin']


- **The cleaned file still mixes in non-regular-season games.** Preseason (about 25.6K rows), All-Star games
  (with team names like `LeBron`, `Giannis`, `Rising Stars`), the play-in, and four spellings of the NBA
  Cup. Averaging over every row would add exhibition games to "regular-season" stats. When aggregating,
  **keep `Regular Season` for regular-season features and `Playoffs` for playoff features**. Decide where
  the play-in and NBA Cup games go (NBA Cup group games count as regular-season games; the championship game
  does not).

In [12]:
playoffs = games[games['gameType'].eq('Playoffs')]
bubble = playoffs[(playoffs['gameDate'] >= '2020-10-01') & (playoffs['gameDate'] < '2020-11-01')]
print('2020 Finals games (Oct 2020) are labeled season:', bubble['season'].unique(),
      '| teams:', sorted(bubble['playerteamName'].unique()), '| rows:', len(bubble))

2020 Finals games (Oct 2020) are labeled season: [2021] | teams: ['Heat', 'Lakers'] | rows: 92


- **Season bug: the 2020 Finals are assigned to the wrong season.** Notebook 02 assigns any game in
  October or later to the next season. The 2019-20 playoffs ran into October 2020 (the bubble), so the
  Lakers' and Heat's Finals games are labeled as **2020-21**. That breaks *playoff games played → playoff
  round* for both finalists, and adds playoff games to a season in which they hadn't played yet. Fix:
  assign seasons from `gameId` (NBA game IDs encode the season) or from a list of opening-night dates,
  like the one notebook 06 already has.
- **Plus-minus is gone.** `plusMinusPoints` is in the raw file but wasn't kept in notebook 02's column
  selection, and #4 doesn't have it either. **No processed table has plus-minus,** even though it's one of
  the three named performance metrics. Add it, and `gameId` (useful for counting games and checking for
  duplicates), to the kept columns.
- `season` 2027 (the 2026-27 season in progress, about 4.7K rows) is in the file. It's outside the window
  and needs to be filtered out.

## 6. League leaders (#4) vs. box scores (#2)

#4 was meant to supply official per-game averages. #2 was meant to fill in the players #4 leaves out.

In [13]:
agg = reg.groupby(['season', 'personId']).agg(gp_box=('points', 'size'), pts_box=('points', 'mean')).reset_index()
agg['season_start'] = agg['season'] - 1
j4 = leaders.merge(agg, left_on=['season', 'player_id'], right_on=['season_start', 'personId'], how='left')
print(f"#4 rows found in #2: {j4['pts_box'].notna().mean():.1%}")
print(f"points per game within 0.15: {((j4['pts'] - j4['pts_box']).abs() <= .15).mean():.1%}, "
      f"games played identical: {(j4['gp'] == j4['gp_box']).mean():.1%}")
cover = pd.DataFrame({'#4 qualified players': leaders.groupby('season').size(),
                      '#2 players with a regular-season game': agg.groupby('season_start').size()}).loc[2014:2025]
cover.T

#4 rows found in #2: 100.0%
points per game within 0.15: 93.7%, games played identical: 82.1%


,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
#4 qualified players,267.0,271.0,271.0,253.0,258.0,261.0,237.0,228.0,245.0,240.0,231.0,234.0
#2 players with a regular-season game,492.0,476.0,485.0,539.0,526.0,528.0,540.0,602.0,539.0,570.0,569.0,582.0


**What we found:** every #4 player-season is in #2, and points per game agree closely, so the two sources
are consistent. #4 covers only about **40-55%** of the players who appear in a season, though, and #2 covers
everyone. Since #2 can recompute every #4 statistic for every player, **we recommend building all
regular-season features from #2** and using #4 only as a validation check. Mixing the two would give
different players' features different definitions. The small games-played gaps (#2 counts slightly
differently, and #4 merges games for players traded mid-season) disappear once we use one source.

## 7. Verdict

| Dataset | Clean on its own? | Ready to merge? | What's blocking it |
|---|---|---|---|
| #1 BBRef salaries | Yes | Mostly | End-year seasons; BBRef team codes. Best use: rebuild 2019-20 payrolls and cross-check #5. |
| #2 Box scores | Mostly | **No** | Mixed game types; 2020 Finals in the wrong season; plus-minus and `gameId` dropped; end-year seasons. |
| #3 Kaggle salaries | Yes | Not needed | Same data as #5 (HoopsHype), ends 2021-22. Keep as a documented check only. |
| #4 League leaders | Yes | Optional | Only qualified players (about 40-55% of each season). Redundant with #2; use it to validate. |
| #5 Salary cap | Yes | Mostly | Leaky `next_season_*` columns; 2019-20 under-covered; legal-name variants; no contract length. |
| #6 Transactions (label) | Yes | Yes | Ready. Join by NBA ID where possible. The 2014 offseason is partial (starts July 2015), but it's outside the window. |

**Overall: the individual cleaning is good. Each dataset has been checked for tidiness, missingness, and
outliers, and the label is solid. The datasets are *not yet* ready to be combined into an ML table.** Fixes,
in priority order:

1. **Harmonize seasons.** Add `season_start` everywhere (`season - 1` for #1 and #2) and join only on it.
2. **Fix #2:** assign seasons by game ID or opening-night dates (the 2020 bubble); keep `plusMinusPoints`
   and `gameId`; split regular season and playoffs, dropping preseason and All-Star games; filter to the window.
3. **Decide the population and contract-years feature without future information.** This is the only blocker
   that needs a **new data source** (contract length as of each season), or a clearly stated approximation.
4. **Drop the leaky columns** (`next_season_primary_team`, `changed_team_next_season`) from anything that
   feeds the model.
5. **Join on the NBA ID where possible** (#2, #4, #6), and add a small name-alias table for the
   #5 ↔ #6 name join (Lou/Louis Williams, Enes Kanter/Freedom, Dennis Schröder/Schroeder, ...).
6. **Handle 2019-20 payrolls** (rebuild from #1, or flag and run a sensitivity check).
7. Add one 30-row team lookup table (BBRef code / NBA code / nickname / full name).
8. Re-compute the baseline trade rate on the final population. The hypothesis' 0.06 is lower than the 7.9%
   we see in #5.

After steps 1, 2, 4, 5 and 7, the combined table can be built (#5 player-season ← #2 regular-season and
playoff aggregates ← #5 team cap tier ← #6 label). Step 3 decides whether the *contract* half of the
hypothesis can be tested honestly.